In [1]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# ==================================================
# PROJECT CONFIG
# ==================================================

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)


# ==================================================
# PROJECT HELPERS
# ==================================================

from minio_config import configure_minio, minio_path


# ==================================================
# SPARK SESSION
# ==================================================

spark = (
    SparkSession.builder
    .appName("NYC Building Risk - Serving Layer")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)

configure_minio(spark)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/12 15:11:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/12 15:11:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/12 15:11:23 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/09/12 15:11:23 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Spark version: 3.4.0
Master: local[2]


In [2]:
# ==================================================
# LOAD GOLD DATA
# ==================================================

dim_building = spark.read.parquet(
    minio_path("gold/data_model/dim_building")
)

dim_property = spark.read.parquet(
    minio_path("gold/data_model/dim_property")
)

building_risk = spark.read.parquet(
    minio_path("gold/building_risk/building_risk_score")
)

property_risk = spark.read.parquet(
    minio_path("gold/property_risk/property_risk_score")
)


print("dim_building:", dim_building.count())
print("dim_property:", dim_property.count())
print("building_risk:", building_risk.count())
print("property_risk:", property_risk.count())

26/09/12 15:11:33 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


dim_building: 197958
dim_property: 858284
building_risk: 197958
property_risk: 858284


In [3]:
print("=== DIM BUILDING ===")
dim_building.printSchema()

print("=== BUILDING RISK ===")
building_risk.printSchema()

print("=== PROPERTY RISK ===")
property_risk.printSchema()

=== DIM BUILDING ===
root
 |-- building_id: string (nullable = true)
 |-- bin: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- resolved_bbl: string (nullable = true)
 |-- current_bbl: string (nullable = true)
 |-- current_address: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- bbl_aliases: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- address_aliases: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- identity_status: string (nullable = true)
 |-- match_method: string (nullable = true)
 |-- match_confidence: string (nullable = true)
 |-- resolution_status: string (nullable = true)

=== BUILDING RISK ===
root
 |-- building_id: string (nullable = true)
 |-- bin: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- current_address: string (nullable = true)
 |-- borough: string (nullable 

In [4]:
# ==================================================
# PREPARE BUILDING RISK FOR SERVING
# ==================================================

building_risk_serving = (
    building_risk
    .select(
        "building_id",

        # Final Building Risk
        "building_risk_score",
        F.col("risk_level").alias("building_risk_level"),

        # Normalized risk components
        "score_311",
        "score_hpd_severity",
        "score_hpd_recency",
        "score_dob_active",
        "score_dob_recency",
        "score_building_age_final",

        # Main operational features
        "complaints_0_30",
        "complaints_31_90",
        "complaints_91_365",

        "hpd_open_class_a",
        "hpd_open_class_b",
        "hpd_open_class_c",
        "hpd_open_class_i",
        "hpd_active_violations",

        "dob_active_violations",

        # Building / property context
        "building_age",
        "age_imputed_flag"
    )
)

In [5]:
# ==================================================
# PREPARE PROPERTY RISK FOR BUILDING SERVING
# ==================================================

property_risk_serving = (
    property_risk
    .select(
        "property_id",

        # Property identity
        "bbl",
        "property_address",
        "zipcode",

        # Final Property Risk
        "property_risk_score",
        "property_risk_level",
        "property_risk_source",

        # Property context
        "building_count",
        "max_building_risk_score",
        "avg_building_risk_score",
        "property_only_risk_score",

        # PLUTO attributes
        "yearbuilt",
        "numbldgs",
        "numfloors",
        "unitsres",
        "unitstotal",
        "landuse",
        "bldgclass",

        "risk_as_of_date"
    )
)

In [6]:
# ==================================================
# BUILD SERVING BUILDING
# Grain: 1 row = 1 building_id
# ==================================================

serving_building = (
    dim_building

    .join(
        building_risk_serving,
        on="building_id",
        how="left"
    )

    .join(
        property_risk_serving,
        on="property_id",
        how="left"
    )
)

In [7]:
serving_building = (
    serving_building
    .select(
        # =========================
        # BUILDING IDENTITY
        # =========================
        "building_id",
        "bin",
        "property_id",

        "current_address",
        "borough",
        "latitude",
        "longitude",

        "resolved_bbl",
        "current_bbl",

        "address_aliases",
        "bbl_aliases",

        # =========================
        # BUILDING RISK
        # =========================
        "building_risk_score",
        "building_risk_level",

        "score_311",
        "score_hpd_severity",
        "score_hpd_recency",
        "score_dob_active",
        "score_dob_recency",
        "score_building_age_final",

        # =========================
        # BUILDING FEATURES
        # =========================
        "complaints_0_30",
        "complaints_31_90",
        "complaints_91_365",

        "hpd_open_class_a",
        "hpd_open_class_b",
        "hpd_open_class_c",
        "hpd_open_class_i",
        "hpd_active_violations",

        "dob_active_violations",

        "building_age",
        "age_imputed_flag",

        # =========================
        # PROPERTY CONTEXT
        # =========================
        "bbl",
        "property_address",
        "zipcode",

        "property_risk_score",
        "property_risk_level",
        "property_risk_source",

        "building_count",
        "max_building_risk_score",
        "avg_building_risk_score",
        "property_only_risk_score",

        # =========================
        # PLUTO
        # =========================
        "yearbuilt",
        "numbldgs",
        "numfloors",
        "unitsres",
        "unitstotal",
        "landuse",
        "bldgclass",

        # =========================
        # MODEL METADATA
        # =========================
        "identity_status",
        "match_method",
        "match_confidence",
        "resolution_status",
        "risk_as_of_date"
    )
)

In [8]:
# ==================================================
# VALIDATE SERVING BUILDING
# ==================================================

print(
    "Serving building rows:",
    serving_building.count()
)

print(
    "Distinct building_id:",
    serving_building
    .select("building_id")
    .distinct()
    .count()
)

print(
    "With Building Risk:",
    serving_building
    .filter(
        F.col("building_risk_score").isNotNull()
    )
    .count()
)

print(
    "With Property:",
    serving_building
    .filter(
        F.col("property_id").isNotNull()
    )
    .count()
)

print(
    "With Property Risk:",
    serving_building
    .filter(
        F.col("property_risk_score").isNotNull()
    )
    .count()
)

Serving building rows: 197958
Distinct building_id: 197958
With Building Risk: 197958
With Property: 197459
With Property Risk: 197459


In [9]:
# ==================================================
# BUILD SERVING PROPERTY
# Grain: 1 row = 1 property_id
# ==================================================

serving_property = (
    property_risk
    .select(
        # =========================
        # PROPERTY IDENTITY
        # =========================
        "property_id",
        "bbl",
        "property_address",
        "borough",
        "zipcode",
        "latitude",
        "longitude",

        # =========================
        # PROPERTY RISK
        # =========================
        "property_risk_score",
        "property_risk_level",
        "property_risk_source",

        # =========================
        # BUILDING CONTEXT
        # =========================
        "building_count",
        "max_building_risk_score",
        "avg_building_risk_score",
        "property_only_risk_score",

        # =========================
        # RISK COMPONENTS
        # =========================
        "property_score_311",
        "property_score_hpd_severity",
        "property_score_hpd_recency",

        # =========================
        # RAW SIGNALS
        # =========================
        "property_risk_311_raw",
        "property_risk_hpd_severity_raw",
        "property_risk_hpd_recency_raw",

        # =========================
        # RECENT 311
        # =========================
        "property_311_0_30",
        "property_311_31_90",
        "property_311_91_365",

        # =========================
        # HPD
        # =========================
        "property_hpd_open_class_a",
        "property_hpd_open_class_b",
        "property_hpd_open_class_c",

        "property_hpd_0_30",
        "property_hpd_31_90",
        "property_hpd_91_365",

        # =========================
        # PLUTO
        # =========================
        "yearbuilt",
        "numbldgs",
        "numfloors",
        "unitsres",
        "unitstotal",
        "landuse",
        "bldgclass",

        # =========================
        # MODEL METADATA
        # =========================
        "risk_as_of_date"
    )
)

In [10]:
# ==================================================
# VALIDATE SERVING PROPERTY
# ==================================================

print(
    "Serving property rows:",
    serving_property.count()
)

print(
    "Distinct property_id:",
    serving_property
    .select("property_id")
    .distinct()
    .count()
)

print(
    "With Property Risk:",
    serving_property
    .filter(
        F.col("property_risk_score").isNotNull()
    )
    .count()
)

print(
    "NO_DATA:",
    serving_property
    .filter(
        F.col("property_risk_level") == "NO_DATA"
    )
    .count()
)

print(
    "With at least one Building:",
    serving_property
    .filter(
        F.col("building_count") > 0
    )
    .count()
)

Serving property rows: 858284
Distinct property_id: 858284
With Property Risk: 184915
NO_DATA: 673369
With at least one Building: 171582


In [11]:
# ==================================================
# ENRICH SERVING BUILDING FOR SEARCH
# ==================================================

serving_building = (
    serving_building

    .withColumn(
        "entity_type",
        F.lit("BUILDING")
    )

    .withColumn(
        "search_address",
        F.upper(
            F.trim(
                F.col("current_address")
            )
        )
    )

    .withColumn(
        "search_aliases",
        F.col("address_aliases")
    )
)

In [12]:
# ==================================================
# ENRICH SERVING PROPERTY FOR SEARCH
# ==================================================

serving_property = (
    serving_property

    .withColumn(
        "entity_type",
        F.lit("PROPERTY")
    )

    .withColumn(
        "search_address",
        F.upper(
            F.trim(
                F.col("property_address")
            )
        )
    )
)

In [13]:
serving_building.select(
    "building_id",
    "entity_type",
    "search_address"
).show(5, truncate=False)

serving_property.select(
    "property_id",
    "entity_type",
    "search_address"
).show(5, truncate=False)

+-----------+-----------+--------------------+
|building_id|entity_type|search_address      |
+-----------+-----------+--------------------+
|BLD:1000000|BUILDING   |300 EAST 83 STREET  |
|BLD:1000003|BUILDING   |10 SOUTH STREET     |
|BLD:1000008|BUILDING   |30 WATER STREET     |
|BLD:1000016|BUILDING   |102 BROAD STREET    |
|BLD:1000018|BUILDING   |1 STATE STREET PLAZA|
+-----------+-----------+--------------------+
only showing top 5 rows

+---------------+-----------+---------------+
|property_id    |entity_type|search_address |
+---------------+-----------+---------------+
|PROP:1000010010|PROPERTY   |140 CARDER ROAD|
|PROP:1000010100|PROPERTY   |CARDER ROAD    |
|PROP:1000010111|PROPERTY   |ANDES ROAD     |
|PROP:1000010150|PROPERTY   |COMFORT ROAD   |
|PROP:1000010201|PROPERTY   |1 ELLIS ISLAND |
+---------------+-----------+---------------+
only showing top 5 rows



In [14]:
# ==================================================
# SERVING QUALITY CHECK
# ==================================================

print("=== BUILDING ===")

print(
    "Missing search_address:",
    serving_building
    .filter(F.col("search_address").isNull())
    .count()
)

print(
    "Missing coordinates:",
    serving_building
    .filter(
        F.col("latitude").isNull()
        | F.col("longitude").isNull()
    )
    .count()
)


print("\n=== PROPERTY ===")

print(
    "Missing search_address:",
    serving_property
    .filter(F.col("search_address").isNull())
    .count()
)

print(
    "Missing coordinates:",
    serving_property
    .filter(
        F.col("latitude").isNull()
        | F.col("longitude").isNull()
    )
    .count()
)

=== BUILDING ===
Missing search_address: 0
Missing coordinates: 183

=== PROPERTY ===
Missing search_address: 577
Missing coordinates: 937


In [15]:
# ==================================================
# SAVE SERVING BUILDING
# ==================================================

SERVING_BUILDING_PATH = minio_path(
    "serving/building"
)

(
    serving_building
    .write
    .mode("overwrite")
    .parquet(SERVING_BUILDING_PATH)
)

print(
    "Saved serving_building to:",
    SERVING_BUILDING_PATH
)

26/09/12 15:18:17 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Saved serving_building to: s3a://nyc-building-risk/serving/building


In [16]:
# ==================================================
# SAVE SERVING PROPERTY
# ==================================================

SERVING_PROPERTY_PATH = minio_path(
    "serving/property"
)

(
    serving_property
    .write
    .mode("overwrite")
    .parquet(SERVING_PROPERTY_PATH)
)

print(
    "Saved serving_property to:",
    SERVING_PROPERTY_PATH
)

Saved serving_property to: s3a://nyc-building-risk/serving/property


In [17]:
saved_building = spark.read.parquet(
    SERVING_BUILDING_PATH
)

saved_property = spark.read.parquet(
    SERVING_PROPERTY_PATH
)

print(
    "Saved building rows:",
    saved_building.count()
)

print(
    "Saved building IDs:",
    saved_building
    .select("building_id")
    .distinct()
    .count()
)

print(
    "Saved property rows:",
    saved_property.count()
)

print(
    "Saved property IDs:",
    saved_property
    .select("property_id")
    .distinct()
    .count()
)

Saved building rows: 197958
Saved building IDs: 197958
Saved property rows: 858284
Saved property IDs: 858284
